
# 01 — OpenAlex Journal Dataset Builder

Journal-level dataset construction for the paper.

This notebook:
- collects articles from selected scientometrics / metrics journals through OpenAlex,
- computes a 3-year citation window,
- creates `filename` values like `1.pdf`, `2.pdf`, ...
- creates journal-aware `citation_bin` labels,
- saves `clean_dataset.csv` for the modeling notebook.

# =========================
# 1. SETUP
# =========================

In [ ]:
import os
import re
import time
import math
import requests
import pandas as pd
import numpy as np
from datetime import datetime

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE_DIR = '/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER' if IN_COLAB else '.'
os.makedirs(BASE_DIR, exist_ok=True)

EMAIL = "kampatzistelis@gmail.com"  # βάλε το email σου για polite OpenAlex usage

RAW_OUT = os.path.join(BASE_DIR, "journal_openalex_raw_dataset.csv")
FULL_OUT = os.path.join(BASE_DIR, "journal_openalex_dataset.csv")
CLEAN_OUT = os.path.join(BASE_DIR, "clean_dataset.csv")

session = requests.Session()
session.headers.update({
    "User-Agent": f"journal-impact-prediction-study (mailto:{EMAIL})"
})

print("BASE_DIR:", BASE_DIR)

# =========================
# 2. CONFIG
# =========================


In [ ]:
JOURNALS = [
    "Scientometrics",                                                     # SCIM
    "Journal of Informetrics",                                            # JOI
    "Quantitative Science Studies",                                       # QSS
    #"Research Evaluation",                                                # RE
    #"Journal of Scientometric Research",                                  # JSR
    "Journal of the Association for Information Science and Technology",  # JASIST
    "Frontiers in Research Metrics and Analytics",                        # RMA
    #"Publications",                                                       # PUBS
    "Iberoamerican Journal of Science Measurement and Communication",     # IJSMC
    "Frontiers in Communication",                                         # FCOMM
]

PUBLICATION_YEARS = list(range(2008, 2023))  # 2008-2022, ώστε το 3-year window να είναι ώριμο.
CITATION_WINDOW_YEARS = 3                    # publication year, year+1, year+2

# Αν ένα journal-year έχει λίγα papers, το binning ανά έτος γίνεται ασταθές.
# Άρα πρώτα προσπαθούμε journal-year, αλλιώς journal-period.
MIN_GROUP_SIZE_FOR_YEAR_BINNING = 30

# Βάζω None για πλήρη συλλογή.
MAX_WORKS_PER_JOURNAL_YEAR = None  # π.χ. 300 για ενα γρήγορο test.

PER_PAGE = 200
SLEEP = 0.15

print("Journals:", JOURNALS)
print("Years:", PUBLICATION_YEARS)

# =========================
# 3. HELPER FUNCTIONS
# =========================

In [ ]:
def clean_title(text):
    if not isinstance(text, str):
        return text

    text = text.replace("&lt;", "<").replace("&gt;", ">").replace("&amp;", "&")

    tags_to_remove = [
        "<i>", "</i>", "<scp>", "</scp>", "<b>", "</b>",
        "<sub>", "</sub>", "<sup>", "</sup>"
    ]

    for tag in tags_to_remove:
        text = text.replace(tag, "")

    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def normalize_doi(doi):
    if pd.isna(doi):
        return None
    doi = str(doi).strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")
    doi = doi.replace("doi.org/", "")
    return doi if doi else None

In [ ]:
def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return None
    word_positions = []
    for word, positions in inverted_index.items():
        for pos in positions:
            word_positions.append((pos, word))
    word_positions.sort()
    return " ".join([word for pos, word in word_positions])

In [ ]:
def get_nested(d, path, default=None):
    cur = d
    for key in path:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
        if cur is None:
            return default
    return cur

In [ ]:
def three_year_citations_from_counts(work, publication_year, window_years=3):
    # counts_by_year έχει citations ανά calendar year.
    # Για 3-year window: publication_year, publication_year+1, publication_year+2
    total = 0
    allowed_years = set(range(publication_year, publication_year + window_years))
    for item in work.get("counts_by_year", []) or []:
        y = item.get("year")
        if y in allowed_years:
            total += item.get("cited_by_count", 0) or 0
    return total

In [ ]:
def source_display_name_from_work(work):
    return (
        get_nested(work, ["primary_location", "source", "display_name"])
        or get_nested(work, ["host_venue", "display_name"])
        or None
    )

In [ ]:
def get_pdf_url(work):
    return (
        get_nested(work, ["primary_location", "pdf_url"])
        or get_nested(work, ["open_access", "oa_url"])
        or None
    )

In [ ]:
def get_landing_page_url(work):
    return (
        get_nested(work, ["primary_location", "landing_page_url"])
        or work.get("doi")
        or work.get("id")
    )

In [ ]:
def period_block(year):
    if year <= 2012:
        return "2008-2012"
    elif year <= 2017:
        return "2013-2017"
    return "2018-2022"

# =========================
# 4. RESOLVE OPENALEX JOURNAL SOURCES
# =========================

In [ ]:
def search_source_by_name(journal_name):
    url = "https://api.openalex.org/sources"

    if journal_name == "Publications":
        params = {
            "filter": "issn:2304-6775",
            "per-page": 10
        }
    else:
        params = {
            "search": journal_name,
            "filter": "type:journal",
            "per-page": 10
        }


    r = session.get(url, params=params, timeout=30)
    r.raise_for_status()
    results = r.json().get("results", [])

    if not results:
        raise ValueError(f"No OpenAlex source found for journal: {journal_name}")

    if journal_name == "Publications":
        selected = results[0]
    # Προτιμάμε exact case-insensitive match.
    else:
        exact = [
            x for x in results
            if str(x.get("display_name", "")).strip().lower() == journal_name.strip().lower()
        ]

        selected = exact[0] if exact else results[0]

    return {
        "query_name": journal_name,
        "source_id": selected.get("id"),
        "source_id_short": str(selected.get("id", "")).split("/")[-1],
        "display_name": selected.get("display_name"),
        "issn_l": selected.get("issn_l"),
        "publisher": selected.get("host_organization_name"),
        "works_count": selected.get("works_count"),
        "cited_by_count": selected.get("cited_by_count")
    }


source_rows = []
for name in JOURNALS:
    try:
        row = search_source_by_name(name)
        source_rows.append(row)
        print("Resolved:", name, "->", row["display_name"], row["source_id_short"])
        time.sleep(SLEEP)
    except Exception as e:
        print("[SOURCE ERROR]", name, e)

sources_df = pd.DataFrame(source_rows)
sources_df.to_csv(os.path.join(BASE_DIR, "openalex_selected_journal_sources.csv"), index=False)
display(sources_df)

# =========================
# 5. FETCH WORKS PER JOURNAL-YEAR
# =========================


In [ ]:
SELECT_FIELDS = ",".join([
    "id",
    "doi",
    "title",
    "display_name",
    "publication_year",
    "type",
    "cited_by_count",
    "counts_by_year",
    "referenced_works_count",
    "primary_topic",
    "primary_location",
    "open_access",
    "abstract_inverted_index",
    "authorships"
])


def fetch_works_for_source_year(source_id_short, journal_name, year):
    url = "https://api.openalex.org/works"
    cursor = "*"
    rows = []
    fetched = 0

    filters = [
        f"primary_location.source.id:{source_id_short}",
        f"publication_year:{year}",
        "type:article",
        "has_doi:true",
        "is_oa:true"
    ]

    while True:
        params = {
            "filter": ",".join(filters),
            "select": SELECT_FIELDS,
            "per-page": PER_PAGE,
            "cursor": cursor
        }

        r = session.get(url, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        results = data.get("results", [])

        if not results:
            break

        for work in results:
            pub_year = work.get("publication_year")
            if pub_year is None:
                continue

            title = clean_title(work.get("title") or work.get("display_name"))
            doi = work.get("doi")
            doi_norm = normalize_doi(doi)

            discipline = "Unknown"
            topic = work.get("primary_topic")

            if topic and topic.get("field"):
                discipline = topic["field"].get("display_name", "Unknown")

            if not title or not doi_norm:
                continue

            rows.append({
                "openalex_id": work.get("id"),
                "doi": doi,
                "doi_norm": doi_norm,
                "title": title,
                "year": pub_year,
                "journal": source_display_name_from_work(work) or journal_name,
                "discipline": discipline,
                "journal_query_name": journal_name,
                "citations": three_year_citations_from_counts(work, pub_year, CITATION_WINDOW_YEARS),
                "total_citations": work.get("cited_by_count", 0),
                "references": work.get("referenced_works_count"),
                "is_oa": get_nested(work, ["open_access", "is_oa"]),
                "oa_status": get_nested(work, ["open_access", "oa_status"]),
                "pdf_url": get_pdf_url(work),
                "landing_page_url": get_landing_page_url(work),
                "abstract": reconstruct_abstract(work.get("abstract_inverted_index")),
            })

            fetched += 1

            if MAX_WORKS_PER_JOURNAL_YEAR is not None and fetched >= MAX_WORKS_PER_JOURNAL_YEAR:
                return rows

        cursor = data.get("meta", {}).get("next_cursor")
        if not cursor:
            break

        time.sleep(SLEEP)

    return rows


all_rows = []

for _, source in sources_df.iterrows():
    source_id_short = source["source_id_short"]
    journal_name = source["display_name"]

    for year in PUBLICATION_YEARS:
        print(f"Fetching {journal_name} / {year}")
        try:
            rows = fetch_works_for_source_year(source_id_short, journal_name, year)
            print("  rows:", len(rows))
            all_rows.extend(rows)
        except Exception as e:
            print("[FETCH ERROR]", journal_name, year, e)

        time.sleep(SLEEP)

raw_df = pd.DataFrame(all_rows)

# Deduplicate DOI
if not raw_df.empty:
    raw_df = raw_df.drop_duplicates(subset=["doi_norm"]).reset_index(drop=True)

raw_df.to_csv(RAW_OUT, index=False)

print("Raw rows:", len(raw_df))
display(raw_df.head())

# =========================
# 6. CLEAN, ADD FILENAME, AND CREATE IMPACT BINS
# =========================

In [ ]:
df = raw_df.copy()

# Basic cleaning
df = df[df["title"].notna()].copy()
df = df[df["title"].astype(str).str.strip().ne("")].copy()
df = df[df["journal"].notna()].copy()
df = df[df["year"].isin(PUBLICATION_YEARS)].copy()

df["title"] = df["title"].apply(clean_title)
df["title_length"] = df["title"].astype(str).apply(lambda x: len(x.split()))
df["has_pdf_url"] = df["pdf_url"].notna().astype(int)
df["has_abstract"] = df["abstract"].notna().astype(int)
df["publication_period"] = df["year"].apply(period_block)

journal_mapping = {
    "Scientometrics": "SCIM",
    "Journal of Informetrics": "JOI",
    "Quantitative Science Studies": "QSS",
    #"Research Evaluation": "RE",
    #"Journal of Scientometric Research": "JSR",
    "Journal of the Association for Information Science and Technology": "JASIST",
    "Frontiers in Research Metrics and Analytics": "RMA",
    #"Publications": "PUBS",
    "Iberoamerican Journal of Science Measurement and Communication": "IJSMC",
    "Frontiers in Communication": "FCOMM",
}

df["journal_short"] = df["journal"].map(journal_mapping)

missing_short = df[df["journal_short"].isna()]["journal"].unique()

if len(missing_short) > 0:
    print("⚠ Journals without abbreviation mapping:")
    print(missing_short)

# Stable order πριν μπει filename.
df = df.sort_values(["journal", "year", "title", "doi_norm"]).reset_index(drop=True)

# filename για downstream merge με rare_citations_summary.csv
df.insert(0, "filename", [f"{i}.pdf" for i in range(1, len(df) + 1)])

print("Rows after cleaning:", len(df))
print("\nJournal-year counts:")
display(df.groupby(["journal", "year"]).size().reset_index(name="N").sort_values(["journal", "year"]))

# =========================
# 7. JOURNAL-AWARE CITATION BINNING
# =========================

In [ ]:
def assign_tertile_bins(group, value_col="citations"):
    group = group.copy()

    if len(group) < 3 or group[value_col].nunique() < 2:
        group["citation_bin"] = np.nan
        return group

    try:
        group["citation_bin"] = pd.qcut(
            group[value_col].rank(method="first"),
            q=3,
            labels=["low", "mid", "high"]
        )
    except Exception:
        group["citation_bin"] = np.nan

    return group


def safe_concat(parts):
    clean_parts = []

    for part in parts:
        if part is None or part.empty:
            continue

        part = part.copy()

        # αφαιρεί columns που είναι 100% NaN για να μη βγάζει FutureWarning το concat
        part = part.dropna(axis=1, how="all")

        clean_parts.append(part)

    if not clean_parts:
        return pd.DataFrame()

    return pd.concat(clean_parts, ignore_index=True)


# Reset bins
df["citation_bin"] = np.nan
df["binning_level"] = np.nan

# 1) Πρώτα journal-year binning
year_parts = []

for _, group in df.groupby(["journal", "year"], sort=False):
    group = group.copy()

    if len(group) >= MIN_GROUP_SIZE_FOR_YEAR_BINNING:
        group = assign_tertile_bins(group, value_col="citations")
        group["binning_level"] = "journal_year"
    else:
        group["citation_bin"] = np.nan
        group["binning_level"] = np.nan

    year_parts.append(group)

df = safe_concat(year_parts)


# 2) Fallback σε journal-period για όσα έμειναν χωρίς bin
stable_part = df[df["citation_bin"].notna()].copy()
fallback_part = df[df["citation_bin"].isna()].copy()

fallback_parts = []

for _, group in fallback_part.groupby(["journal", "publication_period"], sort=False):
    group = group.copy()
    group = assign_tertile_bins(group, value_col="citations")
    group["binning_level"] = "journal_period"
    fallback_parts.append(group)

if fallback_parts:
    fallback_part = safe_concat(fallback_parts)
    df = safe_concat([stable_part, fallback_part])
else:
    df = stable_part.copy()


# Restore filename order
df = df.sort_values(
    "filename",
    key=lambda s: s.str.replace(".pdf", "", regex=False).astype(int)
).reset_index(drop=True)


print("Citation bin counts:")
print(df["citation_bin"].value_counts(dropna=False))

print("\nJournal x citation_bin:")
display(pd.crosstab(df["journal"], df["citation_bin"], dropna=False))

print("\nBinning levels:")
print(df["binning_level"].value_counts(dropna=False))

# =========================
# 8. SAVE FULL DATASET AND CLEAN_DATASET
# =========================

In [ ]:
# Full journal dataset: κρατά όλα τα OpenAlex metadata.
df.to_csv(FULL_OUT, index=False)

# clean_dataset: αυτό θα διαβάζει το modeling notebook.
clean_cols = [
    "filename",
    "openalex_id",
    "doi",
    "title",
    "journal",
    "journal_short",
    "discipline",
    "year",
    "citations",
    "total_citations",
    "citation_bin",
    "references",
    "title_length",
    "is_oa",
    "oa_status",
    "has_pdf_url",
    "pdf_url",
    "landing_page_url",
    "abstract",
    "has_abstract",
    "binning_level",
    "publication_period"
]

clean_cols = [c for c in clean_cols if c in df.columns]
clean_df = df[clean_cols].copy()

# Αφαίρεση εγγραφών που δεν μπόρεσαν να λάβουν έγκυρο citation_bin.
n_before = len(clean_df)
clean_df = clean_df.dropna(subset=["citation_bin"]).reset_index(drop=True)
n_removed = n_before - len(clean_df)

print(f"Rows removed λόγω NaN citation_bin: {n_removed}")
print(f"Final clean_dataset rows: {len(clean_df)}")
print("\nFinal citation_bin counts:")
print(clean_df["citation_bin"].value_counts(dropna=False))

# Τα mid παραμένουν εδώ και θα αφαιρεθούν αργότερα στο modeling notebook.
clean_df.to_csv(CLEAN_OUT, index=False)

# =========================
# 9. BASIC QUALITY REPORT
# =========================


In [ ]:
report = {
    "n_rows": len(clean_df),
    "n_journals": clean_df["journal"].nunique(),
    "years": f"{min(PUBLICATION_YEARS)}-{max(PUBLICATION_YEARS)}",
    "citation_window_years": CITATION_WINDOW_YEARS,
    "n_with_pdf_url": int(clean_df["has_pdf_url"].sum()) if "has_pdf_url" in clean_df else None,
    "pdf_url_rate": float(clean_df["has_pdf_url"].mean()) if "has_pdf_url" in clean_df else None,
    "n_with_abstract": int(clean_df["has_abstract"].sum()) if "has_abstract" in clean_df else None,
    "abstract_rate": float(clean_df["has_abstract"].mean()) if "has_abstract" in clean_df else None,
}

pd.Series(report).to_csv(os.path.join(BASE_DIR, "journal_dataset_quality_report.csv"))
print(report)

print("\nRows per journal:")
display(clean_df["journal"].value_counts())

print("\nRows per journal-year:")
display(clean_df.groupby(["journal", "year"]).size().reset_index(name="N"))

END